<a href="https://colab.research.google.com/github/gyaneshhere/ReInforcementLearning/blob/main/examples/VLM_GRPO_basic_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Example code for VLMGRPOTrainer

Please note that this notebook is intended only as a proof of concept using a simplified dataset. Do not expect meaningful results from this demonstration. Before deploying for your specific use case, you may fine-tune the model on your target data. For proper GRPO training methodology, please refer to the official Unsloth notebooks.

In [ ]:
!pip install --no-deps -U transformers==4.54.1
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install --no-deps -U bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.18.2 cut_cross_entropy unsloth_zoo==2025.7.11
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps -U unsloth==2025.7.11
!pip install -U triton==3.2.0
!pip install --force-reinstall --no-deps git+https://github.com/GAD-cell/vlm-grpo.git@main

In [ ]:
import vlmgrpo
from unsloth import FastVisionModel # FastLanguageModel for LLMs

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0.1,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

In [ ]:
from datasets import load_dataset, Dataset

ds = load_dataset("AI4Math/MathVista",split="testmini")

def is_numeric_answer(example):
  try:
    float(example["answer"])
    return True
  except:
    return False

numeric_ds = ds.filter(is_numeric_answer) # For convenience and easier reward function definition, we keep only the float answer ex : 42

In [ ]:
print(numeric_ds)

Dataset({
    features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
    num_rows: 566
})


# Data loader and reward functions

In [ ]:
reasoning_start = "<think>"
reasoning_end   = "</think>"
solution_start  = "<answer>"
solution_end    = "</answer>"

def format_fn(sample):
    image = sample['decoded_image']
    prompt = sample['question']
    answer = sample['answer']
    format = {"prompt": [
                  {
                  "role": "user",
                  "content": [
                      {"type": "image"}, # because we have only 1 image per sample
                      {"type": "text",  "text": f"{prompt} provide your reasoning between {reasoning_start} and {reasoning_end} and then your final answer between {solution_start} and (put a float here) {solution_end}"}]
                  }],
                  "image": [image.resize((512,512))],
                  "answer":answer,
              }
    return format

class FormattedDataset():
    def __init__(self, dataset, format_fn):
        self.dataset = dataset
        self.format_fn = format_fn

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return self.format_fn(item)

    def __len__(self):
        return len(self.dataset)


train_dataset=FormattedDataset(dataset=numeric_ds,format_fn=format_fn)

In [ ]:
# Reward functions
def formatting_reward_func(completions,**kwargs):
    import re
    print("test")
    thinking_pattern = f'{reasoning_start}(.*?){reasoning_end}'
    answer_pattern = f'{solution_start}(.*?){solution_end}'

    scores=[]
    for completion in completions :
      score=0
      thinking_matches = re.findall(thinking_pattern, completion[0]['content'], re.DOTALL)
      answer_matches = re.findall(answer_pattern, completion[0]['content'], re.DOTALL)
      if len(thinking_matches) == 1 :
        score +=1.0
      if len(answer_matches) == 1 :
        score +=1.0
      scores.append(score)
    return scores


def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    import re

    answer_pattern = f'{solution_start}(.*?){solution_end}'

    responses = [re.findall(answer_pattern, completion[0]['content'], re.DOTALL) for completion in completions]
    q = prompts[0][-1]['content']

    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:{completions[0]}")
    return [2.0 if len(r)==1 and a == r[0].replace('\n','') else 0.0 for r, a in zip(responses, answer)]

# pour contrer le design du GRPO
#def length_reward_func(prompts, completions, answer, **kwargs):


# GRPO Training

In [ ]:
from unsloth import is_bf16_supported
from vlmgrpo import VLMGRPOTrainer
from trl import GRPOConfig

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = is_bf16_supported(),
    fp16 = not is_bf16_supported(),
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 200,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

trainer = VLMGRPOTrainer(
    model = model,
    processing_class=tokenizer, # MUST put unsloth processor here !
    reward_processing_classes = tokenizer, #Here also
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = train_dataset,
    grad_verbose=True
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 566 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 32768, 'temperature': 1e-06, 'repetition_penalty': 1.05, 'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


test
-------------------- Question:
[{'type': 'image'}, {'type': 'text', 'text': 'What is the highest value on the X axis? provide your reasoning between <think> and </think> and then your final answer between <answer> and (put a float here) </answer>'}] 
Answer:
30 
Response:[{'role': 'assistant', 'content': 'What is the highest value on the X axis? provide your reasoning between <think> and </think> and then your final answer between <answer> and (put a float here) </answer>\nassistant\n<think>\nThe X-axis in the graph represents "MICROGRAMS/mL-E-DNP-LYSINE-HCL". The values on the X-axis range from 0 to 30, with increments of 5. The highest value on the X-axis is therefore 30.\n</think>\n<answer>\n30.0\n</answer>'}]


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


[DEBUG] Loss: 2.0194531202832877e-07
[DEBUG] Params with grad: 712 / 1441
[DEBUG] Grad norm: 60.0924


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / formatting_reward_func / mean,rewards / formatting_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.000000,0.000000,0.000000,125.000000,125.000000,125.000000,0.000000,125.000000,125.000000,125.000000,0.000202,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,127.000000,127.000000,127.000000,0.000000,127.000000,127.000000,127.000000,0.000132,0.000000,0.000000,0.000000,0.000000
3,0.000000,1.000000,0.000000,135.000000,135.000000,135.000000,0.000000,135.000000,135.000000,135.000000,0.002632,1.000000,0.000000,0.000000,0.000000
4,0.000000,2.000000,0.000000,133.000000,133.000000,133.000000,0.000000,133.000000,133.000000,133.000000,0.012876,2.000000,0.000000,0.000000,0.000000
5,0.000000,2.000000,0.000000,126.000000,126.000000,126.000000,0.000000,126.000000,126.000000,126.000000,0.002039,2.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,136.000000,136.000000,136.000000,0.000000,136.000000,136.000000,136.000000,0.017239,0.000000,0.000000,0.000000,0.000000
7,0.000100,0.000000,0.000000,124.000000,124.000000,124.000000,0.000000,124.000000,124.000000,124.000000,0.086079,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,150.000000,150.000000,150.000000,0.000000,150.000000,150.000000,150.000000,0.009385,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,127.000000,127.000000,127.000000,0.000000,127.000000,127.000000,127.000000,0.025053,0.000000,0.000000,0.000000,0.000000
10,0.000000,0.000000,0.000000,130.000000,130.000000,130.000000,0.000000,130.000000,130.000000,130.000000,0.026664,0.000000,0.000000,0.000000,0.000000


test
-------------------- Question:
[{'type': 'image'}, {'type': 'text', 'text': 'What is the age gap between these two people in image? provide your reasoning between <think> and </think> and then your final answer between <answer> and (put a float here) </answer>'}] 
Answer:
6 
Response:[{'role': 'assistant', 'content': "What is the age gap between these two people in image? provide your reasoning between <think> and </think> and then your final answer between <answer> and (put a float here) </answer>\nassistant\n<think>\nThe image shows two individuals standing next to each other, but without knowing their exact ages or having additional context, it's impossible to determine the precise age gap between them. The individuals appear to be young adults, possibly in their late teens or early twenties, based on their physical appearance and attire. However, this is just an educated guess and not a definitive statement.\n</think>\n<answer>\n(0.5)\n</answer>"}]
Unsloth: Will smartly offloa

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / formatting_reward_func / mean,rewards / formatting_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.000000,0.000000,0.000000,125.000000,125.000000,125.000000,0.000000,125.000000,125.000000,125.000000,0.000202,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,127.000000,127.000000,127.000000,0.000000,127.000000,127.000000,127.000000,0.000132,0.000000,0.000000,0.000000,0.000000
3,0.000000,1.000000,0.000000,135.000000,135.000000,135.000000,0.000000,135.000000,135.000000,135.000000,0.002632,1.000000,0.000000,0.000000,0.000000
4,0.000000,2.000000,0.000000,133.000000,133.000000,133.000000,0.000000,133.000000,133.000000,133.000000,0.012876,2.000000,0.000000,0.000000,0.000000
5,0.000000,2.000000,0.000000,126.000000,126.000000,126.000000,0.000000,126.000000,126.000000,126.000000,0.002039,2.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,136.000000,136.000000,136.000000,0.000000,136.000000,136.000000,136.000000,0.017239,0.000000,0.000000,0.000000,0.000000
7,0.000100,0.000000,0.000000,124.000000,124.000000,124.000000,0.000000,124.000000,124.000000,124.000000,0.086079,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,150.000000,150.000000,150.000000,0.000000,150.000000,150.000000,150.000000,0.009385,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,127.000000,127.000000,127.000000,0.000000,127.000000,127.000000,127.000000,0.025053,0.000000,0.000000,0.000000,0.000000
10,0.000000,0.000000,0.000000,130.000000,130.000000,130.000000,0.000000,130.000000,130.000000,130.000000,0.026664,0.000000,0.000000,0.000000,0.000000


test
-------------------- Question:
[{'type': 'image'}, {'type': 'text', 'text': 'Use a calculator to find the measure of $∠J$ to the nearest degree. provide your reasoning between <think> and </think> and then your final answer between <answer> and (put a float here) </answer>'}] 
Answer:
40 
Response:[{'role': 'assistant', 'content': 'Use a calculator to find the measure of $∠J$ to the nearest degree. provide your reasoning between <think> and </think> and then your final answer between <answer> and (put a float here) </answer>\nassistant\n<think>\nTo find the measure of ∠J in the right triangle JKL, we can use trigonometric functions. Since ∠L is a right angle, we can use the tangent function, which is defined as the ratio of the opposite side to the adjacent side.\n\nGiven:\n- The length of the hypotenuse JK = 14\n- The length of the side KL = 9\n\nWe need to find the measure of ∠J. In a right triangle, the tangent of an angle is the ratio of the length of the opposite side to the 

In [ ]:
model.save_pretrained("V0_GRPO")  # Local saving
tokenizer.save_pretrained("V0_GRPO")

[]